In [0]:
#Leyendo las tablas de la GEIH
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

In [0]:
# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
cgse

In [0]:
ocupados 

In [0]:
# Calcular la población en edad de trabajar
pet = ft["FEX_C18"].sum()

print("Población en edad de trabajar:", f"{pet:,.0f}".replace(",", "."))

1.1 R/=Para junio de 2026, la población en edad de trabajar estimada fue de 41.202.631 personas

In [0]:
# Calcular la población total
poblacion_total = cgse["FEX_C18"].sum()

print("Población total:", f"{poblacion_total:,.0f}".replace(",", "."))

In [0]:
# Calcular la población que no está en edad de trabajar
poblacion_no_pet = poblacion_total - pet

print("Población que no está en edad de trabajar:",
      f"{poblacion_no_pet:,.0f}".replace(",", "."))

1.2 R/=Para junio de 2026, la población que no estaba en edad de trabajar se estimó en 11.346.181 personas.

In [0]:
# Calcular la población económicamente activa
pea = ft.loc[ft["FT"] == 1, "FEX_C18"].sum()

print("Población económicamente activa:", f"{pea:,.0f}".replace(",", "."))

1.3 R/=Para junio de 2026, la población económicamente activa se estimó en 26.593.429 personas.

In [0]:
# Calcular la población económicamente inactiva
poblacion_inactiva = ft.loc[ft["FFT"] == 1, "FEX_C18"].sum()

print("Población económicamente inactiva:",
      f"{poblacion_inactiva:,.0f}".replace(",", "."))

1.4 R/= Para junio de 2026, la población económicamente inactiva se estimó en 14.609.202 personas.

In [0]:
# Calcular la tasa de ocupación
tasa_ocupacion = ocupados["FEX_C18"].sum() / pet * 100

print("Tasa de ocupación:", f"{tasa_ocupacion:.2f}%")

1.5 R/=Para junio de 2026, la tasa de ocupación fue del 59,36 %.

In [0]:
# Calcular la tasa de desempleo
desempleados_total = desempleados.loc[
    desempleados["DSI"] == 1, "FEX_C18"
].sum()

tasa_desempleo = desempleados_total / pea * 100

print("Tasa de desempleo:", f"{tasa_desempleo:.2f}%")

1.6 R/=Para junio de 2026, la tasa de desempleo fue del 8,03 %.

**PUNTO 2**

In [0]:
import pandas as pd

In [0]:
# Agregar el sexo y la edad a la tabla de desempleados
desempleados = pd.merge(
    desempleados,
    cgse[["DIRECTORIO", "SECUENCIA_P", "ORDEN", "P3271", "P6040"]],
    on=["DIRECTORIO", "SECUENCIA_P", "ORDEN"],
    how="left"
)

In [0]:
# Revisar los códigos de sexo
print(desempleados["P3271"].value_counts(dropna=False))

In [0]:
# Agregar el sexo a la tabla de fuerza de trabajo
ft = pd.merge(
    ft,
    cgse[["DIRECTORIO", "SECUENCIA_P", "ORDEN", "P3271"]],
    on=["DIRECTORIO", "SECUENCIA_P", "ORDEN"],
    how="left"
)

In [0]:
cgse

In [0]:
# Calcular la población económicamente activa por departamento y sexo
pea_sexo = ft[ft["FT"] == 1].groupby(["DPTO", "P3271"])["FEX_C18"].sum()
print(pea_sexo.head())

In [0]:
# Calcular los desempleados por departamento y sexo
desempleo_sexo = (
    desempleados[desempleados["DSI"] == 1]
    .groupby(["DPTO", "P3271"])["FEX_C18"]
    .sum()
)

print(desempleo_sexo.head())

In [0]:
# Calcular la tasa de desempleo por departamento y sexo
tasas_desempleo = (
    (desempleo_sexo / pea_sexo) * 100
).reset_index(name="TASA_DESEMPLEO")

tasas_desempleo["SEXO"] = tasas_desempleo["P3271"].map({
    1: "Hombres",
    2: "Mujeres"
})

print(tasas_desempleo)

In [0]:
# Agregar los nombres de los departamentos
nombres_dpto = {
    5: "Antioquia", 8: "Atlántico", 11: "Bogotá D.C.",
    13: "Bolívar", 15: "Boyacá", 17: "Caldas",
    18: "Caquetá", 19: "Cauca", 20: "Cesar",
    23: "Córdoba", 25: "Cundinamarca", 27: "Chocó",
    41: "Huila", 44: "La Guajira", 47: "Magdalena",
    50: "Meta", 52: "Nariño", 54: "Norte de Santander",
    63: "Quindío", 66: "Risaralda", 68: "Santander",
    70: "Sucre", 73: "Tolima", 76: "Valle del Cauca",
    81: "Arauca", 85: "Casanare", 86: "Putumayo",
    88: "San Andrés", 91: "Amazonas", 94: "Guainía",
    95: "Guaviare", 97: "Vaupés", 99: "Vichada"
}

tasas_desempleo["DEPARTAMENTO"] = tasas_desempleo["DPTO"].map(nombres_dpto)

In [0]:
# tasas de desempleo en tabla 
tabla_sexo = tasas_desempleo.pivot(
    index="DEPARTAMENTO",
    columns="SEXO",
    values="TASA_DESEMPLEO"
).round(2)

# Mostrar la tabla completa con los departamentos
display(tabla_sexo.reset_index())

In [0]:
# Identificar las tasas más altas y más bajas por sexo
for sexo in ["Hombres", "Mujeres"]:
    datos = tasas_desempleo[tasas_desempleo["SEXO"] == sexo]
    mayor = datos.loc[datos["TASA_DESEMPLEO"].idxmax()]
    menor = datos.loc[datos["TASA_DESEMPLEO"].idxmin()]

    print(f"\n{sexo}:")
    print(f"Tasa más alta: {mayor['DEPARTAMENTO']} "
          f"({mayor['TASA_DESEMPLEO']:.2f}%)")
    print(f"Tasa más baja: {menor['DEPARTAMENTO']} "
          f"({menor['TASA_DESEMPLEO']:.2f}%)")

**2R/=**Al comparar las tasas de desempleo por departamento y sexo, se encontró que, entre los hombres, la tasa más alta fue la de San Andrés (20,98 %) y la más baja la de Vaupés (3,00 %). En las mujeres, la tasa más alta correspondió a Putumayo (25,02 %) y la más baja a Vaupés (1,78 %).

**PUNTO 3**

In [0]:
# Revisar las columnas de la tabla de desempleados
print(desempleados.columns.tolist())

In [0]:
# Organizar las edades en grupos
desempleados["GRUPO_EDAD"] = pd.cut(
    desempleados["P6040"],
    bins=[17, 24, 30, 40, 50, 120],
    labels=["18-24", "24-30", "30-40", "40-50", "60+"],
    right=True
)

In [0]:
# Calcular los desempleados por departamento y grupo de edad
desempleo_edad = (
    desempleados[desempleados["DSI"] == 1]
    .groupby(["DPTO", "GRUPO_EDAD"], observed=True)["FEX_C18"]
    .sum()
    .reset_index(name="DESEMPLEADOS")
)

display(desempleo_edad.head(10))

In [0]:
# Agregar la edad a la tabla de fuerza de trabajo
ft = pd.merge(
    ft,
    cgse[["DIRECTORIO", "SECUENCIA_P", "ORDEN", "P6040"]],
    on=["DIRECTORIO", "SECUENCIA_P", "ORDEN"],
    how="left"
)

In [0]:
# Organizar la fuerza de trabajo por grupos de edad
ft["GRUPO_EDAD"] = pd.cut(
    ft["P6040"],
    bins=[17, 24, 30, 40, 50, 120],
    labels=["18-24", "24-30", "30-40", "40-50", "60+"],
    right=True
)

In [0]:
# Calcular la PEA por departamento y grupo de edad
pea_edad = (
    ft[ft["FT"] == 1]
    .groupby(["DPTO", "GRUPO_EDAD"], observed=True)["FEX_C18"]
    .sum()
    .reset_index(name="PEA")
)

display(pea_edad.head(10))

In [0]:
# Calcular la tasa de desempleo por grupo de edad
tasas_edad = pd.merge(
    desempleo_edad,
    pea_edad,
    on=["DPTO", "GRUPO_EDAD"],
    how="inner"
)

tasas_edad["TASA_DESEMPLEO"] = (
    tasas_edad["DESEMPLEADOS"] / tasas_edad["PEA"] * 100
).round(2)

display(tasas_edad.head(10))

In [0]:
# Agregar los nombres de los departamentos
tasas_edad["DEPARTAMENTO"] = tasas_edad["DPTO"].map(nombres_dpto)

In [0]:
# Identificar las tasas más altas y más bajas por departamento
resultados_edad = tasas_edad.loc[
    tasas_edad.groupby("DPTO")["TASA_DESEMPLEO"].idxmax()
].merge(
    tasas_edad.loc[
        tasas_edad.groupby("DPTO")["TASA_DESEMPLEO"].idxmin()
    ][["DPTO", "GRUPO_EDAD", "TASA_DESEMPLEO"]],
    on="DPTO",
    suffixes=("_ALTA", "_BAJA")
)

resultados_edad = resultados_edad[
    ["DEPARTAMENTO", "GRUPO_EDAD_ALTA", "TASA_DESEMPLEO_ALTA",
     "GRUPO_EDAD_BAJA", "TASA_DESEMPLEO_BAJA"]
]

display(resultados_edad)

**3.R/**=Los resultados muestran que el grupo de 18 a 24 años presenta la tasa de desempleo más alta en la mayoría de los departamentos. Se destacan Sucre (32,82 %), San Andrés (30,94 %) y Bolívar (30,66 %). En algunos departamentos, como Chocó, Nariño y Putumayo, la tasa más alta corresponde al grupo de 24 a 30 años.

Por otra parte, el grupo identificado como 60+ presenta la tasa más baja en la mayoría de los departamentos. Se destaca Risaralda, con un 1,12 %, seguido de Huila, con un 1,58 %, y Meta, con un 1,78 %. Lo cual evidencia diferencias en las tasas de desempleo según la edad y el departamento.

**punto 4**

In [0]:
# Calcular las mujeres desempleadas de 18 a 25 años
desempleo_mujeres_edad = (
    desempleados[
        (desempleados["DSI"] == 1) &
        (desempleados["P3271"] == 2) &
        (desempleados["P6040"].between(18, 25))
    ]
    .groupby("DPTO")["FEX_C18"]
    .sum()
    .reset_index(name="DESEMPLEADAS")
)

display(desempleo_mujeres_edad.head())

In [0]:
# Calcular la PEA de mujeres entre 18 y 25 años
pea_mujeres_edad = (
    ft[
        (ft["FT"] == 1) &
        (ft["P3271"] == 2) &
        (ft["P6040"].between(18, 25))
    ]
    .groupby("DPTO")["FEX_C18"]
    .sum()
    .reset_index(name="PEA_MUJERES")
)

display(pea_mujeres_edad.head())

In [0]:
# Calcular la tasa de desempleo de mujeres entre 18 y 25 años
tasas_mujeres_edad = pd.merge(
    desempleo_mujeres_edad,
    pea_mujeres_edad,
    on="DPTO",
    how="inner"
)

tasas_mujeres_edad["TASA_DESEMPLEO"] = (
    tasas_mujeres_edad["DESEMPLEADAS"] /
    tasas_mujeres_edad["PEA_MUJERES"] * 100
).round(2)

tasas_mujeres_edad["DEPARTAMENTO"] = (
    tasas_mujeres_edad["DPTO"].map(nombres_dpto)
)

display(tasas_mujeres_edad) 

In [0]:
# Identificar los tres departamentos con menor tasa de desempleo
tres_departamentos = tasas_mujeres_edad.nsmallest(
    3, "TASA_DESEMPLEO"
)

display(
    tres_departamentos[
        ["DEPARTAMENTO", "TASA_DESEMPLEO"]
    ]
)

**4.R/**=Los tres departamentos con las tasas de desempleo más bajas entre las mujeres de 18 a 25 años fueron Nariño, con 4,29 %; Amazonas, con 4,87 %; y Huila, con 6,58 %.

**PUNTO 5**

In [0]:
# Revisar las columnas de la tabla de ocupados
print(ocupados.columns.tolist())

In [0]:
# Calcular el ingreso laboral promedio por departamento
departamentos = {
    17: "Caldas",
    63: "Quindío",
    66: "Risaralda",
    5: "Antioquia",
    76: "Valle del Cauca"
}

ingresos_promedio = (
    ocupados[ocupados["DPTO"].isin(departamentos.keys())]
    .groupby("DPTO")["INGLABO"]
    .mean()
    .reset_index(name="INGRESO_PROMEDIO")
)

ingresos_promedio["DEPARTAMENTO"] = (
    ingresos_promedio["DPTO"].map(departamentos)
)

display(ingresos_promedio[["DEPARTAMENTO", "INGRESO_PROMEDIO"]])

**5.R/=** Antioquia presentó el ingreso laboral promedio más alto, con $2.754.450, mientras que Valle del Cauca registró el más bajo, con $2.219.018, entre los cinco departamentos analizados.